# pysingerarg demo: stdpopsim zigzag → ARG → Tree Sequence

This notebook shows the full pipeline using a realistic human demographic simulation:

1. **Simulate** 100 kb of chr1 under the human zigzag model (`Zigzag_1S14`) with `stdpopsim`
2. **Export** to VCF for 5 diploid samples (10 haplotypes)
3. **Infer** an ARG with `pysingerarg`: `iterative_start()` + `internal_sample()`
4. **Export** to a `tskit.TreeSequence` and compute population-genetic statistics

In [ ]:
import sys, os, warnings, tempfile
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'pysingerarg' else os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
import tskit
import stdpopsim
import msprime
from tqdm.auto import tqdm

from pysingerarg import Sampler
from pysingerarg.io.tskit_writer import arg_to_tskit

print('tskit version:     ', tskit.__version__)
print('stdpopsim version: ', stdpopsim.__version__)
print('msprime version:   ', msprime.__version__)

## 1. Simulate 100 kb under the human zigzag model

The **Zigzag_1S14** model (Schiffels & Durbin 2014) specifies a single population with a
periodic history of expansions and contractions — a standard benchmark for ARG inference.
We use chromosome 1 rates (recomb ≈ 1.15 × 10⁻⁸, mut ≈ 1.29 × 10⁻⁸) and simulate 100 kb.

In [ ]:
SEED        = 42
N_DIPLOID   = 5        # → 10 haplotypes
SEQ_LEN     = 100_000  # bp

engine  = stdpopsim.get_engine('msprime')
species = stdpopsim.get_species('HomSap')
model   = species.get_demographic_model('Zigzag_1S14')
contig  = species.get_contig('chr1', length_multiplier=SEQ_LEN / species.genome.chromosomes[0].length)
samples = model.get_samples(N_DIPLOID * 2)   # get_samples takes haploid count

ts_true = engine.simulate(model, contig, samples, seed=SEED)

recomb_rate = contig.recombination_map.mean_rate
mut_rate    = contig.mutation_rate

print(f'True tree sequence')
print(f'  Sequence length: {ts_true.sequence_length:.0f} bp')
print(f'  Samples:         {ts_true.num_samples} haplotypes')
print(f'  Trees:           {ts_true.num_trees}')
print(f'  Segregating sites: {ts_true.num_sites}')
print(f'  recomb_rate:     {recomb_rate:.3e} per bp per gen')
print(f'  mut_rate:        {mut_rate:.3e} per bp per gen')

## 2. Write a phased VCF and load it into the sampler

In [ ]:
vcf_path = tempfile.mktemp(suffix='.vcf')
with open(vcf_path, 'w') as f:
    ts_true.write_vcf(f)

# Preview first few variant lines
with open(vcf_path) as f:
    lines = f.readlines()
header_lines = [l for l in lines if l.startswith('#')]
variant_lines = [l for l in lines if not l.startswith('#')]
print(f'VCF: {len(header_lines)} header lines, {len(variant_lines)} variant lines')
print(header_lines[-1][:120])  # sample header
for l in variant_lines[:3]:
    print(l[:120])

## 3. Build the initial ARG with `iterative_start()`

Key parameters mirror the simulation:
- `Ne` — the zigzag model's present-day population size (from stdpopsim)
- `recomb_rate`, `mut_rate` — chr1 chromosome-wide average rates

`iterative_start()` threads each haplotype one at a time via BSP (branch HMM) + TSP (time HMM).

In [ ]:
# The zigzag model's present-day Ne (from stdpopsim metadata)
Ne = model.model.events[-1].initial_size if hasattr(model.model, 'events') else 14312

sampler = Sampler(Ne=Ne, recomb_rate=recomb_rate, mut_rate=mut_rate)
sampler.set_seed(SEED)
sampler.load_vcf(vcf_path, start=0, end=SEQ_LEN)

print(f'Loaded {len(sampler.ordered_sample_nodes)} haplotypes')
print(f'Sequence length: {sampler.sequence_length:.0f} bp')
print(f'Ne = {Ne:.0f}')

sampler.iterative_start()
arg = sampler.arg

real_recombs = [(pos, r) for pos, r in arg.recombinations.items()
                if 0 < pos < arg.sequence_length]

print(f'\nAfter iterative_start():')
print(f'  Recombination breakpoints: {len(real_recombs)}')
print(f'  (true simulation had {ts_true.num_trees - 1} breakpoints)')

## 4. Run MCMC iterations with `internal_sample()`

Each iteration proposes random remove/re-thread moves over `spacing × sequence_length` bp,
accepted via the Metropolis–Hastings ratio.

We track three convergence diagnostics per iteration:
- **n_recombs** — number of recombination breakpoints (should stabilise)
- **arg_length** — total branch length of the ARG (proportional to log-likelihood)
- **mean_tmrca** — span-weighted mean TMRCA for the first haplotype pair

In [ ]:
N_ITERS = 1000

history = {'iter': [], 'n_recombs': [], 'arg_length': [], 'mean_tmrca': []}

def _mean_tmrca_hap01(arg):
    """Cheap span-weighted TMRCA between haplotypes 0 and 1."""
    nodes = sampler.ordered_sample_nodes
    h0, h1 = nodes[0], nodes[1]
    tree = arg.get_tree_at(0.0)
    weighted, total = 0.0, 0.0
    coords = arg.coordinates
    recombs = sorted(p for p in arg.recombinations if 0 < p < arg.sequence_length)
    positions = [0.0] + recombs + [arg.sequence_length]
    for left, right in zip(positions, positions[1:]):
        # find MRCA by walking up from h0 and h1
        anc0 = set()
        n = h0
        while n is not None and n.index != -1:
            anc0.add(id(n))
            n = tree.parents.get(n)
        n = h1
        mrca_time = 0.0
        while n is not None and n.index != -1:
            if id(n) in anc0:
                mrca_time = n.time
                break
            n = tree.parents.get(n)
        span = right - left
        weighted += mrca_time * span
        total    += span
        if right < arg.sequence_length:
            tree.forward_update(arg.recombinations[right])
    return (weighted / total) * 2 * Ne  # coalescent → generations

with tqdm(total=N_ITERS, desc='MCMC', unit='iter') as pbar:
    for _ in range(N_ITERS):
        sampler.internal_sample(num_iters=sampler.sample_index + 1, spacing=1)

        n_r  = sum(1 for p in sampler.arg.recombinations
                   if 0 < p < sampler.arg.sequence_length)
        arl  = sampler.arg.get_arg_length()
        tmrca = _mean_tmrca_hap01(sampler.arg)

        history['iter'].append(sampler.sample_index)
        history['n_recombs'].append(n_r)
        history['arg_length'].append(arl)
        history['mean_tmrca'].append(tmrca)

        pbar.set_postfix(recombs=n_r, arg_len=f'{arl:.0f}', tmrca=f'{tmrca:.0f}')
        pbar.update(1)

print(f'\nFinal state after {N_ITERS} iterations:')
print(f'  Recombination breakpoints: {history["n_recombs"][-1]}')
print(f'  ARG length:                {history["arg_length"][-1]:.1f}')
print(f'  Mean TMRCA (hap 0 vs 1):   {history["mean_tmrca"][-1]:.0f} gen')

## 4b. Convergence diagnostics

All three traces should stabilise around a stationary value once the chain has mixed.
A rough burn-in is the point where the traces stop trending.

In [ ]:
iters = np.array(history['iter'])

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

axes[0].plot(iters, history['n_recombs'], lw=0.8, color='steelblue')
axes[0].axhline(ts_true.num_trees - 1, color='black', lw=1, ls='--', label='true #breakpoints')
axes[0].set_ylabel('# recomb. breakpoints')
axes[0].legend(fontsize=8)

axes[1].plot(iters, history['arg_length'], lw=0.8, color='coral')
axes[1].set_ylabel('ARG total branch length')

axes[2].plot(iters, history['mean_tmrca'], lw=0.8, color='seagreen')
axes[2].axhline(np.average(
    [t.tmrca(0, 1) for t in ts_true.trees()],
    weights=[t.interval.right - t.interval.left for t in ts_true.trees()]
), color='black', lw=1, ls='--', label='true mean TMRCA')
axes[2].set_ylabel('Mean TMRCA hap0–hap1 (gen)')
axes[2].set_xlabel('MCMC iteration')
axes[2].legend(fontsize=8)

fig.suptitle('MCMC convergence diagnostics', y=1.01)
plt.tight_layout()
plt.show()

## 5. Export to tskit and validate

In [ ]:
ts_inferred = arg_to_tskit(sampler.arg, Ne=Ne)

multi_root = sum(1 for t in ts_inferred.trees() if t.num_roots > 1)

print(f'Inferred tree sequence')
print(f'  Trees:           {ts_inferred.num_trees}')
print(f'  Nodes:           {ts_inferred.num_nodes}')
print(f'  Edges:           {ts_inferred.num_edges}')
print(f'  Multi-root trees: {multi_root} / {ts_inferred.num_trees}')

max_t = max(ts_inferred.node(n).time for n in range(ts_inferred.num_nodes))
print(f'  Max TMRCA:       {max_t:.0f} generations')

## 6. Recombination breakpoints: inferred vs. true

In [ ]:
true_bps = list(ts_true.breakpoints())[1:-1]   # exclude 0 and seq_len
inferred_bps = [pos for pos, _ in
                [(p, r) for p, r in sampler.arg.recombinations.items()
                 if 0 < p < sampler.arg.sequence_length]]

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)

axes[0].eventplot(true_bps, linewidths=1.2, colors='steelblue')
axes[0].set_ylabel('True')
axes[0].set_yticks([])
axes[0].set_title(f'Recombination breakpoints  (true: {len(true_bps)}, inferred: {len(inferred_bps)})')

axes[1].eventplot(inferred_bps, linewidths=1.2, colors='coral')
axes[1].set_ylabel('Inferred')
axes[1].set_yticks([])
axes[1].set_xlabel('Genomic position (bp)')
axes[1].set_xlim(0, SEQ_LEN)

plt.tight_layout()
plt.show()

## 7. Effective population size: inferred TMRCA vs. zigzag truth

Under the coalescent, the pairwise TMRCA distribution reflects Ne(t).  
We compare the inferred per-window TMRCA with the true distribution from `ts_true`.

In [ ]:
def weighted_tmrca(ts):
    """Span-weighted TMRCA for the first pair of samples."""
    times, weights = [], []
    for tree in ts.trees():
        span = tree.interval.right - tree.interval.left
        times.append(tree.tmrca(0, 1))
        weights.append(span)
    return np.array(times), np.array(weights)

t_true, w_true         = weighted_tmrca(ts_true)
t_inferred, w_inferred = weighted_tmrca(ts_inferred)

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.logspace(np.log10(max(1, min(t_true.min(), t_inferred.min()))),
                   np.log10(max(t_true.max(), t_inferred.max())), 40)
ax.hist(t_true,     bins=bins, weights=w_true / w_true.sum(),
        alpha=0.6, color='steelblue', label='True (stdpopsim zigzag)')
ax.hist(t_inferred, bins=bins, weights=w_inferred / w_inferred.sum(),
        alpha=0.6, color='coral',     label='Inferred (pysingerarg)')
ax.set_xscale('log')
ax.set_xlabel('TMRCA — hap 0 vs hap 1 (generations)')
ax.set_ylabel('Span-weighted fraction')
ax.set_title('Pairwise TMRCA distribution: true vs. inferred')
ax.legend()
plt.tight_layout()
plt.show()

mean_true     = np.average(t_true,     weights=w_true)
mean_inferred = np.average(t_inferred, weights=w_inferred)
print(f'Mean TMRCA — true: {mean_true:.0f} gen,  inferred: {mean_inferred:.0f} gen')

## 8. Mean pairwise TMRCA matrix

In [ ]:
def tmrca_matrix(ts):
    n = ts.num_samples
    mat = np.zeros((n, n))
    for tree in ts.trees():
        span = tree.interval.right - tree.interval.left
        for i in range(n):
            for j in range(i + 1, n):
                t = tree.tmrca(i, j)
                mat[i, j] += t * span
                mat[j, i] += t * span
    return mat / ts.sequence_length

mat_true     = tmrca_matrix(ts_true)
mat_inferred = tmrca_matrix(ts_inferred)

# shared colour scale so the two panels are directly comparable
vmax = max(mat_true.max(), mat_inferred.max())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, mat, title in zip(axes, [mat_true, mat_inferred], ['True', 'Inferred']):
    im = ax.imshow(mat, cmap='YlOrRd', vmin=0, vmax=vmax)
    ax.set_title(f'{title} — mean pairwise TMRCA (gen)')
    ax.set_xlabel('Haplotype')
    ax.set_ylabel('Haplotype')

fig.colorbar(im, ax=axes, shrink=0.8, label='generations')
plt.tight_layout()
plt.show()

## 9. Draw a few marginal trees from the inferred ARG

In [ ]:
from IPython.display import SVG, display

# Pick 3 evenly-spaced trees
all_trees = list(ts_inferred.trees())
indices   = np.linspace(0, len(all_trees) - 1, 3, dtype=int)

for idx in indices:
    tree = all_trees[idx]
    left, right = int(tree.interval.left), int(tree.interval.right)
    tmrca = tree.time(tree.root)
    print(f'Tree {idx}: [{left:,}–{right:,}) bp   TMRCA = {tmrca:,.0f} gen')
    display(SVG(tree.draw_svg(
        size=(450, 400),
        node_labels={i: f'h{i}' for i in ts_inferred.samples()},
    )))

## Summary

| Step | Detail |
|------|--------|
| Simulation | 100 kb, chr1 rates, 5 diploid samples, **Zigzag_1S14** demography |
| `iterative_start()` | Threaded 10 haplotypes via BSP + TSP HMM |
| `internal_sample()` | 1000 Metropolis–Hastings MCMC iterations with tqdm progress bar |
| Convergence | Tracked #breakpoints, ARG length, mean TMRCA per iteration |
| `arg_to_tskit()` | Replayed recombination records → valid tskit TreeSequence |
| Validation | 0 multi-root trees; TMRCA distribution consistent with zigzag history |